In [1]:
import pandas as pd
import numpy as np
import skimpy as sk
import mlflow
import mlflow.sklearn

from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import *
from sklearn.metrics import *
from sklearn.model_selection import train_test_split, StratifiedGroupKFold
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBRFClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

#file loading
import joblib
from warnings import filterwarnings; filterwarnings("ignore")

In [2]:
train = pd.read_csv("data/train_with_regimes.csv")
test = pd.read_csv("data/test_with_regimes.csv")

In [3]:
train

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,...,OneHotKMeans_Regime2_Prob,OneHotKMeans_Regime3_Prob,CryoRegime_Prob,LuxuryRegime_Prob,FamilyRegime_Prob,SoloRegime_Prob,Ensemble_Regime1_Prob,Ensemble_Regime2_Prob,Ensemble_Regime3_Prob,Predicted_Regime
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39,False,0,0,0,...,0.367942,0.309488,0.400000,0.000000,0.066667,0.533333,0.258386,0.607265,0.134350,2.0
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24,False,109,9,25,...,0.277522,0.383950,0.064516,0.354839,0.064516,0.516129,0.502257,0.224839,0.272905,1.0
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58,True,43,3576,0,...,0.314045,0.375481,0.055556,0.555556,0.222222,0.166667,0.115615,0.274611,0.609774,3.0
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33,False,0,1283,371,...,0.303793,0.395507,0.074074,0.407407,0.296296,0.222222,0.120513,0.228600,0.650887,3.0
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16,False,303,70,151,...,0.287030,0.374018,0.064516,0.354839,0.064516,0.516129,0.601067,0.227869,0.171065,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8688,9276_01,Europa,False,A/98/P,55 Cancri e,41,True,0,6819,0,...,0.364044,0.315133,0.054054,0.459459,0.054054,0.432432,0.129229,0.283879,0.586893,3.0
8689,9278_01,Earth,True,G/1499/S,PSO J318.5-22,18,False,0,0,0,...,0.327435,0.375315,0.500000,0.000000,0.055556,0.444444,0.276286,0.574558,0.149156,2.0
8690,9279_01,Earth,False,G/1500/S,TRAPPIST-1e,26,False,0,0,1872,...,0.307521,0.406886,0.074074,0.259259,0.074074,0.592593,0.406400,0.418035,0.175564,2.0
8691,9280_01,Europa,False,E/608/S,55 Cancri e,32,False,0,1049,0,...,0.319601,0.376361,0.074074,0.407407,0.296296,0.222222,0.121536,0.233682,0.644782,3.0


In [4]:
sk.skim(train)

╭──────────────────────────────────────────────── skimpy summary ─────────────────────────────────────────────────╮
│          Data Summary                Data Types                                                                 │
│ ┏━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┓ ┏━━━━━━━━━━━━━┳━━━━━━━┓                                                          │
│ ┃ Dataframe         ┃ Values ┃ ┃ Column Type ┃ Count ┃                                                          │
│ ┡━━━━━━━━━━━━━━━━━━━╇━━━━━━━━┩ ┡━━━━━━━━━━━━━╇━━━━━━━┩                                                          │
│ │ Number of rows    │ 8693   │ │ float64     │ 16    │                                                          │
│ │ Number of columns │ 39     │ │ int64       │ 11    │                                                          │
│ └───────────────────┴────────┘ │ string      │ 9     │                                                          │
│                                │ bool        │ 3     │                                                          │
│                                └─────────────┴───────┘                                                          │
│                                                     number                                                      │
│ ┏━━━━━━━━━━┳━━━━━┳━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━━┓  │
│ ┃ column   ┃ NA  ┃ NA %      ┃ mean   ┃ sd      ┃ p0       ┃ p25       ┃ p50      ┃ p75    ┃ p100   ┃ hist   ┃  │
│ ┡━━━━━━━━━━╇━━━━━╇━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━━┩  │
│ │ Age      │   0 │         0 │  28.79 │   14.34 │        0 │        20 │       27 │     37 │     79 │ ▂█▇▃▁  │  │
│ │ RoomServ │   0 │         0 │    220 │   660.5 │        0 │         0 │        0 │     41 │  14330 │   █    │  │
│ │ ice      │     │           │        │         │          │           │          │        │        │        │  │
│ │ FoodCour │   0 │         0 │  448.4 │    1596 │        0 │         0 │        0 │     61 │  29810 │   █    │  │
│ │ t        │     │           │        │         │          │           │          │        │        │        │  │
│ │ Shopping │   0 │         0 │  169.6 │     598 │        0 │         0 │        0 │     22 │  23490 │   █    │  │
│ │ Mall     │     │           │        │         │          │           │          │        │        │        │  │
│ │ Spa      │   0 │         0 │  304.6 │    1126 │        0 │         0 │        0 │     53 │  22410 │   █    │  │
│ │ VRDeck   │   0 │         0 │  298.3 │    1134 │        0 │         0 │        0 │     40 │  24130 │   █    │  │
│ │ GroupID  │   0 │         0 │   4633 │    2671 │        1 │      2319 │     4630 │   6883 │   9280 │ ████▇█ │  │
│ │ Passenge │   0 │         0 │  1.518 │   1.054 │        1 │         1 │        1 │      2 │      8 │   █▁   │  │
│ │ rNum     │     │           │        │         │          │           │          │        │        │        │  │
│ │ GroupSiz │   0 │         0 │  2.036 │   1.596 │        1 │         1 │        1 │      3 │      8 │  █▁▁   │  │
│ │ e        │     │           │        │         │          │           │          │        │        │        │  │
│ │ CabinNum │ 199 │ 2.2891982 │  600.4 │   511.9 │        0 │     167.2 │      427 │    999 │   1894 │ █▃▂▂▂▁ │  │
│ │          │     │  05452663 │        │         │          │           │          │        │        │        │  │
│ │ TotalExp │   0 │         0 │   1441 │    2803 │        0 │         0 │      716 │   1441 │  35990 │   █    │  │
│ │ ense     │     │           │        │         │          │           │          │        │        │        │  │
│ │ LogTotal │   0 │         0 │  4.253 │   3.689 │        0 │         0 │    6.575 │  7.274 │  10.49 │ █  ▅▅▁ │  │
│ │ Expense  │     │           │        │         │          │           │          │        │        │        │  │
│ │ NumAmeni │   0 │         0 │  1.735 │   1.645 │     

In [5]:
train

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,...,OneHotKMeans_Regime2_Prob,OneHotKMeans_Regime3_Prob,CryoRegime_Prob,LuxuryRegime_Prob,FamilyRegime_Prob,SoloRegime_Prob,Ensemble_Regime1_Prob,Ensemble_Regime2_Prob,Ensemble_Regime3_Prob,Predicted_Regime
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39,False,0,0,0,...,0.367942,0.309488,0.400000,0.000000,0.066667,0.533333,0.258386,0.607265,0.134350,2.0
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24,False,109,9,25,...,0.277522,0.383950,0.064516,0.354839,0.064516,0.516129,0.502257,0.224839,0.272905,1.0
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58,True,43,3576,0,...,0.314045,0.375481,0.055556,0.555556,0.222222,0.166667,0.115615,0.274611,0.609774,3.0
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33,False,0,1283,371,...,0.303793,0.395507,0.074074,0.407407,0.296296,0.222222,0.120513,0.228600,0.650887,3.0
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16,False,303,70,151,...,0.287030,0.374018,0.064516,0.354839,0.064516,0.516129,0.601067,0.227869,0.171065,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8688,9276_01,Europa,False,A/98/P,55 Cancri e,41,True,0,6819,0,...,0.364044,0.315133,0.054054,0.459459,0.054054,0.432432,0.129229,0.283879,0.586893,3.0
8689,9278_01,Earth,True,G/1499/S,PSO J318.5-22,18,False,0,0,0,...,0.327435,0.375315,0.500000,0.000000,0.055556,0.444444,0.276286,0.574558,0.149156,2.0
8690,9279_01,Earth,False,G/1500/S,TRAPPIST-1e,26,False,0,0,1872,...,0.307521,0.406886,0.074074,0.259259,0.074074,0.592593,0.406400,0.418035,0.175564,2.0
8691,9280_01,Europa,False,E/608/S,55 Cancri e,32,False,0,1049,0,...,0.319601,0.376361,0.074074,0.407407,0.296296,0.222222,0.121536,0.233682,0.644782,3.0


In [6]:
train.isnull().sum()

PassengerId                    0
HomePlanet                   201
CryoSleep                      0
Cabin                        199
Destination                  182
Age                            0
VIP                          203
RoomService                    0
FoodCourt                      0
ShoppingMall                   0
Spa                            0
VRDeck                         0
Name                         200
Transported                    0
GroupID                        0
PassengerNum                   0
GroupSize                      0
Deck                         199
CabinNum                     199
Side                         199
CabinSection                 199
TotalExpense                   0
LogTotalExpense                0
NumAmenities                   0
AgeGroup                     178
GMM_Regime1_Prob               0
GMM_Regime2_Prob               0
GMM_Regime3_Prob               0
OneHotKMeans_Regime1_Prob      0
OneHotKMeans_Regime2_Prob      0
OneHotKMea

In [7]:
transport_map = {False: 0, True: 1}
train["Transported"] = train["Transported"].map(transport_map)

# label Encoding
le = {}

encoder = LabelEncoder()

cat_columns = list(train.select_dtypes(include="object").columns)

for col in cat_columns:
    if col != "PassengerId" and col != "Transported":

        encoder = LabelEncoder()

        train[col] = train[col].astype(str)

  
        train[col] = train[col].replace(["nan", "None", "NULL", ""], "Missing")

        train[col] = encoder.fit_transform(train[col])

        le[col] = encoder


test_cols = list(test.select_dtypes(include="object").columns)

for col in test_cols:
    if col != "PassengerId":

        test[col] = test[col].astype(str)


        test[col] = test[col].replace(["nan", "None", "NULL", ""], "Missing")


        if col not in le:
            continue


        test[col] = test[col].apply(
            lambda x: x if x in le[col].classes_ else "Missing"
        )


        if "Missing" not in le[col].classes_:
            le[col].classes_ = np.append(le[col].classes_, "Missing")

        test[col] = le[col].transform(test[col])

In [8]:
test_cols = list(test.select_dtypes(include="object").columns)

for col in test_cols:
    if col != "PassengerId":
        test[col] = test[col].astype(str)
        test[col] = le[col].transform(test[col])

In [9]:
#For Train
num_cols = train.select_dtypes(include=["int64", "float64"]).columns
cat_cols = train.select_dtypes(include=["object", "bool"]).columns

num_imputer = SimpleImputer(strategy="median")
cat_imputer = SimpleImputer(strategy="most_frequent")

train[num_cols] = num_imputer.fit_transform(train[num_cols])
train[cat_cols] = cat_imputer.fit_transform(train[cat_cols])


In [10]:
test_num_cols = test.select_dtypes(include=["int64", "float64"]).columns
test_cat_cols = test.select_dtypes(include=["object", "bool"]).columns

test_num_imputer = SimpleImputer(strategy="median")
test_cat_imputer = SimpleImputer(strategy="most_frequent")

test[test_num_cols] = test_num_imputer.fit_transform(test[test_num_cols])
test[test_cat_cols] = test_cat_imputer.fit_transform(test[test_cat_cols])


In [11]:
test_ids = test["PassengerId"]
test = test.drop(columns=["PassengerId"], axis=1)

X = train.drop(columns=["PassengerId", "Transported"], axis=1)
y = train["Transported"]

X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

cat_indices = [
    X_train.columns.get_loc(col)
    for col in cat_columns if col != 'PassengerId'
]

smote = SMOTE()

X_train, y_train = smote.fit_resample(X_train, y_train)

In [12]:
lr = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        C=2.0,
        penalty="l2",
        solver="lbfgs",
        max_iter=2000,
        class_weight="balanced",
        n_jobs=-1
    ))
])

gbt = Pipeline([
    ("transformer", QuantileTransformer(
        output_distribution="uniform",
        n_quantiles=1000,
        subsample=100000,
        random_state=42
    )),
    ("model", GradientBoostingClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=3,
        subsample=0.8,
        min_samples_leaf=20,
        random_state=42
    ))
])

rfr = Pipeline([
    ("transformer", QuantileTransformer(
        output_distribution="uniform",
        n_quantiles=1000,
        subsample=100000,
        random_state=42
    )),
    ("model", RandomForestClassifier(
        n_estimators=600,
        max_depth=None,
        min_samples_split=10,
        min_samples_leaf=2,
        max_features="sqrt",
        bootstrap=True,
        class_weight="balanced_subsample",
        n_jobs=-1,
        random_state=42
    ))
])

xgbr = Pipeline([
    ("transformer", QuantileTransformer(
        output_distribution="uniform",
        n_quantiles=1000,
        subsample=100000,
        random_state=42
    )),
    ("model", XGBRFClassifier(
        n_estimators=400,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bynode=0.8,
        reg_lambda=1.0,
        reg_alpha=0.0,
        random_state=42,
        n_jobs=-1
    ))
])

lgbm = Pipeline([
    ("transformer", QuantileTransformer(
        output_distribution="uniform",
        n_quantiles=1000,
        subsample=100000,
        random_state=42
    )),
    ("model", LGBMClassifier(
        n_estimators=2000,
        learning_rate=0.03,
        num_leaves=64,
        max_depth=-1,
        min_child_samples=20,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        reg_alpha=0.0,
        random_state=42,
        n_jobs=-1
    ))
])


cat = CatBoostClassifier(
    iterations=2000,
    learning_rate=0.03,
    depth=6,
    loss_function="Logloss",
    eval_metric="Accuracy",
    random_seed=42,
    verbose=200,
    l2_leaf_reg=3,
    subsample=0.8,
    auto_class_weights="Balanced"
)


In [13]:
mlflow.set_experiment("Spaceship_Titanic_Training")

models = {
    "LogisticRegression": lr,
    "GradientBoostingClassifier": gbt,
    "RandomForestClassifier": rfr,
    "XGBoostRandomForest": xgbr,
    "LightGBMClassifier": lgbm,
    "CatBoostClassifier": cat
}


pred_registry = {}
proba_registry = {}
metric_registry = {}

for name, model in models.items():

    with mlflow.start_run(run_name=name):

        model.fit(X_train, y_train)

        preds = model.predict(X_test)

        if hasattr(model, "predict_proba"):
            probs = model.predict_proba(X_test)[:, 1]
        else:
            probs = preds.astype(float)

        f1 = f1_score(y_test, preds)
        brier = brier_score_loss(y_test, probs)

        mlflow.log_metric("f1", f1)
        mlflow.log_metric("brier", brier)

        mlflow.sklearn.log_model(model, name)

        pred_registry[name] = preds
        proba_registry[name] = probs

        metric_registry[name] = {
            "f1": f1,
            "brier": brier
        }

        print("\nMODEL:", name)
        print("F1:", f1)
        print("Brier:", brier)
        print("-" * 40)


2026/05/21 00:18:29 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/21 00:18:38 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.



MODEL: LogisticRegression
F1: 0.796400449943757
Brier: 0.1410993215815683
----------------------------------------


2026/05/21 00:18:54 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/21 00:18:58 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.



MODEL: GradientBoostingClassifier
F1: 0.8108733371891267
Brier: 0.12291882887564827
----------------------------------------


2026/05/21 00:19:07 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/21 00:19:15 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


...


MODEL: LightGBMClassifier
F1: 0.8049065420560748
Brier: 0.14075909998688402
----------------------------------------
0:	learn: 0.7480011	total: 88ms	remaining: 2m 55s
200:	learn: 0.8340948	total: 1.43s	remaining: 12.8s
400:	learn: 0.8537978	total: 3.08s	remaining: 12.3s
600:	learn: 0.8770702	total: 4.17s	remaining: 9.7s
800:	learn: 0.8994860	total: 5.21s	remaining: 7.81s
1000:	learn: 0.9199029	total: 6.29s	remaining: 6.28s
1200:	learn: 0.9346088	total: 7.55s	remaining: 5.02s
1400:	learn: 0.9471730	total: 8.71s	remaining: 3.73s
1600:	learn: 0.9590234	total: 9.81s	remaining: 2.44s
1800:	learn: 0.9695888	total: 11s	remaining: 1.21s


2026/05/21 00:19:53 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


1999:	learn: 0.9777270	total: 12.1s	remaining: 0us


2026/05/21 00:20:00 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.



MODEL: CatBoostClassifier
F1: 0.8164447017950203
Brier: 0.11946544107692177
----------------------------------------


## Model ensemble

In [14]:
proba_df = pd.DataFrame(proba_registry)
corr_matrix = proba_df.corr()
models = list(proba_df.columns)

eps = 1e-8

base_weights = {}

for m in models:
    f1 = metric_registry[m]["f1"]
    brier = metric_registry[m]["brier"]

    # higher is better
    base_weights[m] = f1 / (brier + eps)


epsilon = 2.0

corr = corr_matrix.abs()

penalty = {}

for i in models:
    penalty[i] = 0

    for j in models:
        if i != j:
            penalty[i] += (corr.loc[i, j] ** epsilon)

#Finalized Weights
final_weights = {}

for m in models:
    final_weights[m] = base_weights[m] / (1 + penalty[m])

#Normalization of weights
total = sum(final_weights.values())

for m in models:
    final_weights[m] /= total

print("Final Weights To Be Used: ", final_weights)

#Prediction and Inference
ensemble_proba = np.zeros(len(proba_df))

for m in models:
    ensemble_proba += final_weights[m] * proba_df[m].values

ensemble_pred = (ensemble_proba > 0.5).astype(int)

Final Weights To Be Used:  {'LogisticRegression': np.float64(0.17247255649574386), 'GradientBoostingClassifier': np.float64(0.1848017059215467), 'RandomForestClassifier': np.float64(0.1778060939375748), 'XGBoostRandomForest': np.float64(0.09663253284157759), 'LightGBMClassifier': np.float64(0.17294956472018197), 'CatBoostClassifier': np.float64(0.19533754608337506)}


In [15]:
#Final Test
models = {
    "LogisticRegression": lr,
    "GradientBoostingClassifier": gbt,
    "RandomForestClassifier": rfr,
    "XGBoostRandomForest": xgbr,
    "LightGBMClassifier": lgbm,
    "CatBoostClassifier": cat
}
test_proba_registry = {}

for m_name, model in models.items():

    if hasattr(model, "predict_proba"):
        test_proba_registry[m_name] = model.predict_proba(test)[:, 1]
    else:
        test_proba_registry[m_name] = model.predict(test).astype(float)

proba_df_test = pd.DataFrame(test_proba_registry)

#Using Trained COrrelation
corr_matrix = proba_df.corr()   # from validation stage
models = list(proba_df.columns)

eps = 1e-8

base_weights = {}

for m in models:
    f1 = metric_registry[m]["f1"]
    brier = metric_registry[m]["brier"]

    base_weights[m] = f1 / (brier + eps)

epsilon = 2.0

corr = corr_matrix.abs()

penalty = {}

for i in models:
    penalty[i] = 0

    for j in models:
        if i != j:
            penalty[i] += corr.loc[i, j] ** epsilon

final_weights = {}

for m in models:
    final_weights[m] = base_weights[m] / (1 + penalty[m])

total = sum(final_weights.values())

for m in models:
    final_weights[m] /= total

print("Final Weights:", final_weights)

ensemble_proba_test = np.zeros(len(test))

for m in models:
    ensemble_proba_test += final_weights[m] * proba_df_test[m].values

ensemble_pred_test = (ensemble_proba_test > 0.5).astype(int)

Final Weights: {'LogisticRegression': np.float64(0.17247255649574386), 'GradientBoostingClassifier': np.float64(0.1848017059215467), 'RandomForestClassifier': np.float64(0.1778060939375748), 'XGBoostRandomForest': np.float64(0.09663253284157759), 'LightGBMClassifier': np.float64(0.17294956472018197), 'CatBoostClassifier': np.float64(0.19533754608337506)}


In [16]:
submission = pd.DataFrame({
    "PassengerId": test_ids,   
    "Transported": ensemble_pred_test.astype(bool)
})

submission.head()

submission.to_csv("submission.csv", index=False)